In [ ]:
# | default_exp wuilt.cli

# Wuilt CLI
> CLI commands for Wuilt store sync and SEO optimization.

In [ ]:
#| export
from fastcore.script import call_parse
from seootter.sqlite_db import get_session
from seootter.models import get_wuilt_products, get_wuilt_pages, print_wuilt_stores, print_wuilt_products, print_wuilt_pages
from seootter.wuilt.sync import sync_wuilt_products, sync_wuilt_pages, sync_wuilt_collections, sync_wuilt_store
from dotenv import load_dotenv
import os

load_dotenv()


True

In [ ]:
#| export
@call_parse
def seo_otter_wuilt_sync(
    store_id: str = "",  # Wuilt store ID (defaults to WUILT_STORE_ID env var)
    api_key: str = "",  # Wuilt API key (defaults to WUILT_API_KEY env var)
    store_name: str = "",  # Optional store display name
    locale: str = "ar",  # Store locale
):
    "Sync products from Wuilt API to local database"
    store_id = store_id or os.getenv("WUILT_STORE_ID")
    api_key = api_key or os.getenv("WUILT_API_KEY")
    if not store_id or not api_key:
        print("❌ Provide --store-id and --api-key or set WUILT_STORE_ID and WUILT_API_KEY in .env")
        return
    result = sync_wuilt_products(
        api_key=api_key,
        store_id=store_id,
        store_name=store_name or store_id,
        locale=locale,
    )
    print(f"✅ Synced: {result['total']} products ({result['created']} new, {result['updated']} updated)")

In [ ]:
#| export
@call_parse
def seo_otter_wuilt_list(
    store_id: str = "",  # Optional: filter by store ID from env
):
    "List synced Wuilt products from local database"
    with get_session() as session:
        products = get_wuilt_products(session)
    if store_id:
        products = [p for p in products if p.store_id == store_id]
    print_wuilt_products(products)

In [ ]:
#| export
@call_parse
def seo_otter_wuilt_store(
    store_id: str = "",
    api_key: str = "",
):
    "Fetch and display store info (SEO + robots.txt) from Wuilt"
    store_id = store_id or os.getenv("WUILT_STORE_ID")
    api_key = api_key or os.getenv("WUILT_API_KEY")
    if not store_id or not api_key:
        print("Provide --store-id and --api-key or set env vars")
        return
    from seootter.wuilt.client import WuiltClient
    client = WuiltClient(api_key=api_key, store_id=store_id)
    store = client.get_store()
    seo = store.get('seo') or {}
    robots = store.get('robotsTxt') or {}
    print(f'Store: {store.get("name")}')
    print(f'  SEO Title: {seo.get("title") or "-"}')
    print(f'  SEO Desc:  {seo.get("description") or "-"}')
    content = robots.get('content') or ''
    print(f'  Robots: {len(content)} chars')
    if content:
        print(content[:300])


In [ ]:
#| export
@call_parse
def seo_otter_wuilt_pages(
    store_id: str = "",
    api_key: str = "",
    locale: str = "ar",
):
    "Sync Wuilt store pages to local database"
    store_id = store_id or os.getenv("WUILT_STORE_ID")
    api_key = api_key or os.getenv("WUILT_API_KEY")
    if not store_id or not api_key:
        print("Provide --store-id and --api-key or set env vars")
        return
    result = sync_wuilt_pages(
        api_key=api_key, store_id=store_id,
        store_name=store_id, locale=locale,
    )
    print(f'Pages: {result["total"]} ({result["created"]} new, {result["updated"]} updated)')


In [ ]:
#| export
@call_parse
def seo_otter_wuilt_optimize(
    store_id: str = "",  # Wuilt store ID
    api_key: str = "",  # Wuilt API key
    locale: str = "ar",  # Store locale
    product_ids: str = "",  # Comma-separated product IDs (omit for all)
    dry_run: bool = False,  # Print optimized fields without pushing
):
    "Generate optimized SEO fields for Wuilt products using DSPy"
    store_id = store_id or os.getenv("WUILT_STORE_ID")
    api_key = api_key or os.getenv("WUILT_API_KEY")
    if not store_id or not api_key:
        print("❌ Provide --store-id and --api-key or set WUILT_STORE_ID and WUILT_API_KEY in .env")
        return

    from seootter.wuilt.optimizer import batch_optimize_products, optimize_product
    from seootter.wuilt.client import WuiltClient

    ids = [i.strip() for i in product_ids.split(",") if i.strip()] if product_ids else None

    if dry_run:
        client = WuiltClient(api_key=api_key, store_id=store_id, locale=locale)
        products = client.get_products()
        if ids:
            products = [p for p in products if p["id"] in ids]
        for p in products:
            print(f"\n=== {p['title']} ({p['id']}) ===")
            seo = p.get("seo") or {}
            optimized = optimize_product(
                product_title=p.get("title", ""),
                description_html=p.get("descriptionHtml", ""),
                short_description=p.get("shortDescription", ""),
                language=locale,
            )
            print(f"  SEO Title:     {seo.get('title')}")
            print(f"  Optimized:     {optimized['seo_title']}\n")
            print(f"  SEO Desc:      {seo.get('description')}")
            print(f"  Optimized:     {optimized['seo_description']}\n")
    else:
        results = batch_optimize_products(
            api_key=api_key,
            store_id=store_id,
            locale=locale,
            product_ids=ids,
        )
        success = sum(1 for r in results if "error" not in r)
        failed = sum(1 for r in results if "error" in r)
        print(f"\n✅ {success} optimized, ❌ {failed} failed")

In [ ]:
# | hide
# | eval: false
# Test: Sync products
seo_otter_wuilt_sync()

# Test: List products
from seootter.sqlite_db import get_session
from seootter.models import get_wuilt_products, print_wuilt_products

with get_session() as session:
    products = get_wuilt_products(session, wuilt_store_id=os.getenv("WUILT_STORE_ID"))
print(f"Total in DB: {len(products)}")
print_wuilt_products(products[:3])

✅ Synced: 52 products (0 new, 52 updated)
Total in DB: 0


No products synced yet.